# Modelling panel — EDA and preparation

Panel-level cleaning and prep on the supervised **pathogen-long** units from notebook `10` (B2/A3 contract).

| Item | Detail |
|------|--------|
| Input | `data/03_processed/modelling_panel.parquet` + `modelling_panel_manifest.json` |
| Grain | Observed VectorNet × pathogen: Tier 1 `(h3_cell, year, month, vector_group, pathogen_name)`; Tier 2 adds `country_iso2` |
| Primary target | `pathogen_within_5y` (forward ≤5y, same key × **this** pathogen) |
| Secondary / QA | `pathogen_within_10y`, ever / delay / same-month; source & geo shares for this pathogen |
| Features (vector) | Group aggregates + `species_richness` + `vector_count_<slug>` priority taxa |
| Training filter (upstream) | VectorNet coverage required — virus-only units already excluded; no synthetic empty months |
| Spec | [`specs/03`](specs/03-spatial-temporal-matching-strategy.md) + [`specs/04`](specs/04-reference-data-and-panel-pipeline.md) |

**Prerequisite:** run [`notebooks/01_sources/10_merge_modelling_panel.ipynb`](notebooks/01_sources/10_merge_modelling_panel.ipynb) so the processed panel exists.

This notebook covers load → profile → missingness / outliers → geo/time filters → modelling handoff. Stratify EDA by `pathogen_name`. Distribution plots, target deep-dive, feature importance, and model training come in later cells.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from vector_guard.data import default_processed_dir, find_repo_root
from vector_guard.resolve import H3_RESOLUTION, load_priority_vector_taxa

REPO_ROOT = find_repo_root(Path.cwd())
PROCESSED_DIR = default_processed_dir(REPO_ROOT)
PANEL_PATH = PROCESSED_DIR / "modelling_panel.parquet"
MANIFEST_PATH = PROCESSED_DIR / "modelling_panel_manifest.json"

# Modelling window: denser VectorNet era (inclusive).
YEAR_MIN = 2010
YEAR_MAX = None  # inclusive upper bound; None = no cap

# Winsorize heavy-tailed vector effort proxies at this quantile.
COUNT_WINSOR_Q = 0.99
COUNT_COLS = [
    "vector_detection_count",
    "individual_count_sum",
    "individual_count_mean",
]

IDENTITY_COLS = [
    "spatial_tier",
    "h3_cell",
    "country_iso2",
    "year",
    "month",
    "vector_group",
    "pathogen_name",
]
TIER1_KEY = ["h3_cell", "year", "month", "vector_group", "pathogen_name"]
TIER2_KEY = ["country_iso2", "year", "month", "vector_group", "pathogen_name"]

# Primary supervised target (spec 04).
TARGET_COL = "pathogen_within_5y"
LABEL_COLS = [
    "pathogen_within_5y",
    "pathogen_within_10y",
    "pathogen_ever_before",
    "pathogen_ever_after",
    "pathogen_same_month",
    "pathogen_detection_count_within_5y",
    "pathogen_detection_count_within_10y",
    "years_to_first_pathogen_after",
    "years_since_last_pathogen_before",
]

# Priority taxon count columns (closed list from reference CSV).
_priority = load_priority_vector_taxa()
PRIORITY_COUNT_COLS = [f"vector_count_{slug}" for slug in _priority["feature_slug"].tolist()]

FEATURE_COLS = [
    "vector_detection_count",
    "individual_count_sum",
    "individual_count_mean",
    "species_richness",
    *PRIORITY_COUNT_COLS,
    "pathogen_ever_before",  # history before row time; not the forward target
]

# Forward-window QA columns (not modelling features — would leak label horizon).
SHARE_COLS = [
    "share_pathogen_exact_coords",
    "share_pathogen_subnational_centroid",
]

{
    "REPO_ROOT": str(REPO_ROOT),
    "PANEL_PATH": str(PANEL_PATH.relative_to(REPO_ROOT)),
    "H3_RESOLUTION": H3_RESOLUTION,
    "YEAR_MIN": YEAR_MIN,
    "n_priority_features": len(PRIORITY_COUNT_COLS),
}


## Load panel and merge manifest

In [ ]:
if not PANEL_PATH.exists():
    raise FileNotFoundError(
        f"Missing {PANEL_PATH}. Run notebooks/01_sources/10_merge_modelling_panel.ipynb first."
    )

panel = pd.read_parquet(PANEL_PATH)
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8")) if MANIFEST_PATH.exists() else {}

missing_priority = [c for c in PRIORITY_COUNT_COLS if c not in panel.columns]
assert not missing_priority, missing_priority
assert "pathogen_name" in panel.columns
assert TARGET_COL in panel.columns
assert "virus_within_5y" not in panel.columns

{
    "rows": len(panel),
    "cols": len(panel.columns),
    "manifest_panel_rows": manifest.get("panel_rows"),
    "manifest_date_range": manifest.get("date_range"),
    "primary_label": manifest.get("primary_label"),
    "secondary_label": manifest.get("secondary_label"),
    "eligibility": manifest.get("eligibility"),
    "n_pathogens": manifest.get("n_pathogens"),
    "pathogen_positive_units": manifest.get("pathogen_positive_units"),
    "pathogen_positive_rate": manifest.get("pathogen_positive_rate"),
    "pathogen_within_10y_units": manifest.get("pathogen_within_10y_units"),
    "pathogen_ever_after_units": manifest.get("pathogen_ever_after_units"),
    "pathogen_ever_before_units": manifest.get("pathogen_ever_before_units"),
    "pathogen_same_month_units": manifest.get("pathogen_same_month_units"),
    "rows_by_spatial_tier": manifest.get("rows_by_spatial_tier"),
    "training_filter": manifest.get("training_filter"),
}


## Panel profile

Shape, dtypes, grain uniqueness, and basic value ranges.

In [ ]:
display(panel.dtypes.to_frame("dtype"))
display(panel.head(5))

tier1 = panel.loc[panel["spatial_tier"] == "h3"]
tier2 = panel.loc[panel["spatial_tier"] == "country"]

profile = {
    "n_rows": len(panel),
    "n_tier1_h3": len(tier1),
    "n_tier2_country": len(tier2),
    "dup_tier1_keys": int(tier1.duplicated(TIER1_KEY).sum()) if len(tier1) else 0,
    "dup_tier2_keys": int(tier2.duplicated(TIER2_KEY).sum()) if len(tier2) else 0,
    "year_min": int(panel["year"].min()),
    "year_max": int(panel["year"].max()),
    "n_h3_cells": int(panel["h3_cell"].nunique(dropna=True)),
    "n_countries": int(panel["country_iso2"].nunique(dropna=True)),
    "n_pathogens": int(panel["pathogen_name"].nunique()),
    "vector_groups": sorted(panel["vector_group"].dropna().unique().tolist()),
    "vector_present_all_one": bool((panel["vector_present"] == 1).all()),
    "pathogen_positive_5y": int(panel[TARGET_COL].sum()),
    "pathogen_positive_rate_5y": float(panel[TARGET_COL].mean()),
    "pathogen_within_10y": int(panel["pathogen_within_10y"].sum()),
    "pathogen_ever_after": int(panel["pathogen_ever_after"].sum()),
    "pathogen_ever_before": int(panel["pathogen_ever_before"].sum()),
    "pathogen_same_month": int(panel["pathogen_same_month"].sum()),
    "species_richness_mean": float(panel["species_richness"].mean()),
}
profile


In [ ]:
display(
    panel.groupby("pathogen_name", dropna=False)
    .agg(
        n_rows=("vector_present", "size"),
        within_5y=(TARGET_COL, "sum"),
        within_10y=("pathogen_within_10y", "sum"),
        ever_after=("pathogen_ever_after", "sum"),
        ever_before=("pathogen_ever_before", "sum"),
        same_month=("pathogen_same_month", "sum"),
        mean_vector_detections=("vector_detection_count", "mean"),
    )
    .assign(positive_rate_5y=lambda d: d["within_5y"] / d["n_rows"])
    .sort_values("positive_rate_5y", ascending=False)
)

display(
    panel.groupby(["vector_group", "pathogen_name"], dropna=False)
    .agg(
        n_rows=("vector_present", "size"),
        within_5y=(TARGET_COL, "sum"),
        within_10y=("pathogen_within_10y", "sum"),
    )
    .assign(positive_rate_5y=lambda d: d["within_5y"] / d["n_rows"])
)

display(
    panel.groupby("year", dropna=False)
    .agg(
        n_rows=("vector_present", "size"),
        within_5y=(TARGET_COL, "sum"),
        within_10y=("pathogen_within_10y", "sum"),
        ever_after=("pathogen_ever_after", "sum"),
    )
    .tail(20)
)


## Missing values

Identity, vector features, and binary labels should be complete on the supervised panel.

Expected nulls:
- `share_pathogen_*` — only defined when the 5y forward window has events of **this** pathogen
- `years_to_first_pathogen_after` / `years_since_last_pathogen_before` — null when that side has no event (`pathogen_ever_after` / `pathogen_ever_before` == 0)


In [ ]:
missing = (
    panel.isna()
    .sum()
    .rename("n_missing")
    .to_frame()
    .assign(pct=lambda d: 100.0 * d["n_missing"] / len(panel))
    .sort_values("n_missing", ascending=False)
)
display(missing.head(20))

share_qa = {
    "share_null_when_pathogen_0": float(
        panel.loc[panel[TARGET_COL] == 0, SHARE_COLS[0]].isna().mean()
    ),
    "share_null_when_pathogen_1": float(
        panel.loc[panel[TARGET_COL] == 1, SHARE_COLS[0]].isna().mean()
    ),
    "delay_after_null_when_ever_after_0": bool(
        panel.loc[panel["pathogen_ever_after"] == 0, "years_to_first_pathogen_after"].isna().all()
    ),
    "delay_before_null_when_ever_before_0": bool(
        panel.loc[panel["pathogen_ever_before"] == 0, "years_since_last_pathogen_before"].isna().all()
    ),
}
share_qa


## Outliers (vector effort proxies)

Detection and individual counts are right-skewed. Summarize tails and prepare winsor caps for modelling features (do not drop units — rare virus positives must stay).

In [ ]:
outlier_summary = []
winsor_caps: dict[str, float] = {}
for col in COUNT_COLS:
    s = panel[col]
    cap = float(s.quantile(COUNT_WINSOR_Q))
    winsor_caps[col] = cap
    outlier_summary.append(
        {
            "column": col,
            "min": float(s.min()),
            "p50": float(s.median()),
            "p95": float(s.quantile(0.95)),
            f"p{int(COUNT_WINSOR_Q * 100)}": cap,
            "max": float(s.max()),
            "n_above_cap": int((s > cap).sum()),
            "n_pos_above_cap": int(
                ((panel[TARGET_COL] == 1) & (s > cap)).sum()
            ),
        }
    )

display(pd.DataFrame(outlier_summary))
winsor_caps

## Geo / time filters and modelling prep

Applied rules:

1. **Identity completeness** — drop rows missing required key fields for their `spatial_tier` (including `pathogen_name`).
2. **Valid calendar month** — `month` in 1–12.
3. **VectorNet coverage** — keep `vector_present == 1` (already true for the merge output).
4. **Modelling year window** — `year >= YEAR_MIN` (and `<= YEAR_MAX` if set).
5. **Feature prep** — winsorize vector count columns at p99. Forward-window `share_pathogen_*` stay nullable QA (not in `FEATURE_COLS`). Delay columns stay nullable and are not modelling features here.

Row drops are reported; primary-label (`pathogen_within_5y`) retention is checked. Stratify later EDA by `pathogen_name`.


In [ ]:
def apply_panel_prep(
    df: pd.DataFrame,
    *,
    year_min: int = YEAR_MIN,
    year_max: int | None = YEAR_MAX,
    winsor_q: float = COUNT_WINSOR_Q,
) -> tuple[pd.DataFrame, dict]:
    """Filter and prepare the modelling panel; return (prepared_df, filter_report)."""
    work = df.copy()
    n0 = len(work)
    n_pos0 = int(work[TARGET_COL].sum())
    report: dict = {"n_input": n0, "n_pos_input": n_pos0}

    # 1. Identity by tier
    h3_ok = (work["spatial_tier"] == "h3") & work["h3_cell"].notna()
    country_ok = (work["spatial_tier"] == "country") & work["country_iso2"].notna()
    common_ok = (
        work["spatial_tier"].isin(["h3", "country"])
        & work["year"].notna()
        & work["month"].notna()
        & work["vector_group"].fillna("").ne("")
        & work["pathogen_name"].fillna("").ne("")
        & work["country_iso2"].notna()  # metadata preferred on Tier 1 too
    )
    mask_id = common_ok & (h3_ok | country_ok)
    report["dropped_identity"] = int((~mask_id).sum())
    work = work.loc[mask_id].copy()

    # 2. Valid month
    mask_month = work["month"].between(1, 12)
    report["dropped_bad_month"] = int((~mask_month).sum())
    work = work.loc[mask_month].copy()

    # 3. VectorNet coverage
    mask_vec = work["vector_present"] == 1
    report["dropped_no_vector"] = int((~mask_vec).sum())
    work = work.loc[mask_vec].copy()

    # 4. Year window
    mask_year = work["year"] >= year_min
    if year_max is not None:
        mask_year &= work["year"] <= year_max
    report["dropped_year_window"] = int((~mask_year).sum())
    report["year_min"] = year_min
    report["year_max"] = year_max
    work = work.loc[mask_year].copy()

    # 5. Feature prep (winsorize count cols)
    caps = {c: float(work[c].quantile(winsor_q)) for c in COUNT_COLS}
    report["winsor_caps"] = caps
    report["winsor_q"] = winsor_q
    for col, cap in caps.items():
        work[f"{col}_raw"] = work[col]
        work[col] = work[col].clip(upper=cap)

    work["year"] = work["year"].astype(int)
    work["month"] = work["month"].astype(int)

    report["n_output"] = len(work)
    report["n_pos_output"] = int(work[TARGET_COL].sum())
    report["n_dropped_total"] = n0 - len(work)
    report["pos_retained"] = report["n_pos_output"] == n_pos0 or report[
        "n_pos_output"
    ] == int(df.loc[df["year"] >= year_min, TARGET_COL].sum())
    return work.reset_index(drop=True), report


panel_prep, filter_report = apply_panel_prep(panel)
filter_report


In [ ]:
prep_check_cols = IDENTITY_COLS + FEATURE_COLS + [
    TARGET_COL,
    "pathogen_within_10y",
    "pathogen_detection_count_within_5y",
    "pathogen_detection_count_within_10y",
    "pathogen_same_month",
]
prep_missing = panel_prep[prep_check_cols].isna().sum()
assert int(prep_missing.sum()) == 0, prep_missing[prep_missing > 0].to_dict()
assert "virus_within_5y" not in panel_prep.columns
assert "virus_detected" not in panel_prep.columns

{
    "prep_rows": len(panel_prep),
    "prep_pos_5y": int(panel_prep[TARGET_COL].sum()),
    "prep_pos_rate_5y": float(panel_prep[TARGET_COL].mean()),
    "prep_within_10y": int(panel_prep["pathogen_within_10y"].sum()),
    "prep_ever_after": int(panel_prep["pathogen_ever_after"].sum()),
    "prep_n_pathogens": int(panel_prep["pathogen_name"].nunique()),
    "prep_year_range": [int(panel_prep["year"].min()), int(panel_prep["year"].max())],
    "prep_missing_feature_cells": int(prep_missing.sum()),
    "spatial_tier_counts": panel_prep["spatial_tier"].value_counts().to_dict(),
}


## Modelling handoff

`panel_prep` is the cleaned analysis table for later EDA / training cells.

| Object | Role |
|--------|------|
| `panel` | Raw merge output (unchanged) |
| `panel_prep` | Filtered + winsorized table |
| `TARGET_COL` | `pathogen_within_5y` |
| `LABEL_COLS` | Forward / ever / delay / same-month label family (pathogen-specific) |
| `FEATURE_COLS` | Vector effort + species richness + priority taxon counts + `pathogen_ever_before` |
| `filter_report` | Drop counts and winsor caps |

Use `pathogen_within_10y` as an alternate target for sensitivity runs. Stratify metrics by `pathogen_name`. Delay columns are for EDA histograms (`.dropna()`), not the primary supervised target.

Next (still Planned on the roadmap): feature distributions, target class-balance analysis (per pathogen), feature importance, spatial/temporal plots, then multi-model training.


In [ ]:
X = panel_prep[FEATURE_COLS].copy()
y = panel_prep[TARGET_COL].astype(int).copy()
meta = panel_prep[
    IDENTITY_COLS
    + [
        "vector_present",
        "first_vector_year",
        "first_vector_month",
        "pathogen_detection_count_within_5y",
        "pathogen_detection_count_within_10y",
        "pathogen_within_10y",
        "pathogen_ever_before",
        "pathogen_ever_after",
        "pathogen_same_month",
        "years_to_first_pathogen_after",
        "years_since_last_pathogen_before",
    ]
].copy()

handoff = {
    "X_shape": X.shape,
    "y_shape": y.shape,
    "y_positive": int(y.sum()),
    "y_positive_rate": float(y.mean()),
    "n_pathogens": int(meta["pathogen_name"].nunique()),
    "feature_cols": FEATURE_COLS,
    "target_col": TARGET_COL,
    "label_cols": LABEL_COLS,
    "filter_report": filter_report,
}
display(X[COUNT_COLS + ["species_richness", "pathogen_ever_before"]].describe().T)
handoff
